# run — BMIA-Adaptive V3: Higher Accuracy at Normal Operating lr

## Goal
Improve accuracy at lr=0.01–0.05 (normal operating range) while keeping zero collapses.

## 2 Approved Changes over BMIA-A (both 6-professor approved)
1. **λ_init = 0.1** (was 0.5) — V17.1 proved ZERO collapses at lr≤0.1 with λ=0.5; MI term alone prevents collapse (CSD Part-b). Lower λ gives more adaptation budget.
2. **Gradient accumulation (ACCUM=2, effective batch=128)** — better H(Y) marginal estimate over 128 samples vs 64. H(Y) gradient is more accurate → better MI optimization.

## What is NOT changed
- No EMA in loss (would break gradient graph — Prof 1 rejected)
- Same ResNet-18, same CIFAR-100-C, same collapse criterion
- Same λ_min=0.01, λ_max=10.0, τ=0.10, momentum=0.9

## Setup
1. Add `rojanregmi1/cifar100-c` via Add Data
2. Add `fedesoriano/cifar100` via Add Data
3. GPU T4 x2, Internet ON

## Runs
- Main: TENT / BMIA-A (old) / BMIA-A-V3 × 4 lr × 5 corruptions = 60 runs
- Ablation: 4 variants (base/accum/lambda/both) × 2 lr × 5 corruptions = 40 runs
- Total: 100 runs, ~35 min

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import time, json, copy, gc, os, pickle
from PIL import Image
from torch.utils.data import Dataset

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

NUM_CLASSES = 100
TTA_BATCH   = 64    # per-accumulation-step batch size
ACCUM       = 2     # gradient accumulation steps → effective batch = 128
EVAL_BATCH  = 256   # evaluation only
N_SAMPLES   = 5000
SEVERITY    = 5
LR_LIST     = [0.005, 0.01, 0.02, 0.05]   # normal operating range
CORRUPTIONS = ['gaussian_noise', 'defocus_blur', 'snow', 'contrast', 'jpeg_compression']
METHODS     = ['tent', 'bmia_adaptive', 'bmia_adaptive_v3']
ABL_LR_LIST = [0.01, 0.05]   # ablation at two representative lr

print(f'TTA effective batch size: {TTA_BATCH * ACCUM}')
print(f'LR_LIST = {LR_LIST}')
print(f'Methods = {METHODS}')
print(f'Main runs: {len(METHODS)}×{len(LR_LIST)}×{len(CORRUPTIONS)} = {len(METHODS)*len(LR_LIST)*len(CORRUPTIONS)}')
print(f'Ablation: 4 variants × {len(ABL_LR_LIST)} lr × {len(CORRUPTIONS)} corruptions = {4*len(ABL_LR_LIST)*len(CORRUPTIONS)}')

In [ ]:
# ================================================================
# Step 1: Train ResNet-18 (seed=42, same architecture as all prior experiments)
# ================================================================
class CIFAR100Pickle(Dataset):
    """Load CIFAR-100 directly from pickle — works with any folder name."""
    def __init__(self, file_path, transform=None):
        with open(file_path, 'rb') as f:
            d = pickle.load(f, encoding='bytes')
        self.imgs   = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        self.labels = np.array(d[b'fine_labels'], dtype=np.int64)
        self.transform = transform
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = Image.fromarray(self.imgs[idx])
        if self.transform: img = self.transform(img)
        return img, int(self.labels[idx])

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

# Auto-detect CIFAR-100 pickle files
CIFAR100_TRAIN_FILE = None
CIFAR100_TEST_FILE  = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'train' in files and 'test' in files and 'meta' in files:
        CIFAR100_TRAIN_FILE = os.path.join(root, 'train')
        CIFAR100_TEST_FILE  = os.path.join(root, 'test')
        print(f'Found CIFAR-100 at: {root}')
        break

if CIFAR100_TRAIN_FILE is None:
    print('Not found — downloading via torchvision...')
    trainset = torchvision.datasets.CIFAR100('/tmp/cifar100', train=True,  download=True, transform=train_transform)
    testset  = torchvision.datasets.CIFAR100('/tmp/cifar100', train=False, download=True, transform=test_transform)
else:
    trainset = CIFAR100Pickle(CIFAR100_TRAIN_FILE, transform=train_transform)
    testset  = CIFAR100Pickle(CIFAR100_TEST_FILE,  transform=test_transform)
    print(f'Loaded: {len(trainset)} train / {len(testset)} test')

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

model = torchvision.models.resnet18(weights=None)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(512, NUM_CLASSES)
model = model.to(device)

optimizer_train = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler       = optim.lr_scheduler.CosineAnnealingLR(optimizer_train, T_max=50)
criterion       = nn.CrossEntropyLoss()

print('Training ResNet-18 (50 epochs)...')
for epoch in range(50):
    model.train()
    for X, y in trainloader:
        X, y = X.to(device), y.to(device)
        optimizer_train.zero_grad()
        criterion(model(X), y).backward()
        optimizer_train.step()
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1}/50')

testloader = torch.utils.data.DataLoader(testset, batch_size=EVAL_BATCH, shuffle=False)
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for X, y in testloader:
        X, y = X.to(device), y.to(device)
        correct += (model(X).argmax(1) == y).sum().item()
        total   += y.size(0)
clean_acc = correct / total
print(f'\nClean accuracy: {clean_acc:.4f}')
assert clean_acc > 0.70, f'Clean acc {clean_acc:.4f} too low — training failed'

SOURCE_STATE = copy.deepcopy(model.state_dict())
print('Source state saved.')

In [ ]:
# ================================================================
# Step 2: Load CIFAR-100-C
# ================================================================
DATA_PATH = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'labels.npy' in files and 'gaussian_noise.npy' in files:
        DATA_PATH = root
        break
assert DATA_PATH is not None, 'CIFAR-100-C not found! Add rojanregmi1/cifar100-c dataset.'
print(f'CIFAR-100-C at: {DATA_PATH}')

missing = [c for c in CORRUPTIONS
           if not os.path.exists(os.path.join(DATA_PATH, f'{c}.npy'))]
assert len(missing) == 0, f'Missing corruption files: {missing}'
print(f'All {len(CORRUPTIONS)} corruption files verified.')

ALL_LABELS = np.load(os.path.join(DATA_PATH, 'labels.npy'))
CIFAR_MEAN = torch.tensor([0.5071, 0.4867, 0.4408]).view(1, 3, 1, 1)
CIFAR_STD  = torch.tensor([0.2675, 0.2565, 0.2761]).view(1, 3, 1, 1)

def load_corruption(name, severity=5, n_samples=N_SAMPLES):
    data  = np.load(os.path.join(DATA_PATH, f'{name}.npy'))
    start = (severity - 1) * 10000
    imgs  = data[start:start + 10000][:n_samples]
    lbls  = ALL_LABELS[start:start + 10000][:n_samples]
    X = torch.from_numpy(imgs.copy()).permute(0, 3, 1, 2).float() / 255.0
    X = (X - CIFAR_MEAN) / CIFAR_STD
    return X.to(device), torch.from_numpy(lbls.copy()).long().to(device)

print('Data loading ready.')

In [ ]:
# ================================================================
# Step 3: Helpers
# ================================================================
def freeze_except_bn(mdl):
    bn_ids = set()
    for m in mdl.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            for p in m.parameters():
                bn_ids.add(id(p))
    for p in mdl.parameters():
        p.requires_grad_(id(p) in bn_ids)

def get_metrics(mdl, X, y):
    mdl.eval()
    all_preds, all_probs = [], []
    with torch.no_grad():
        for i in range(0, X.size(0), EVAL_BATCH):
            logits = mdl(X[i:i+EVAL_BATCH])
            probs  = torch.softmax(logits, 1)
            all_preds.append(logits.argmax(1))
            all_probs.append(probs)
    preds  = torch.cat(all_preds)
    probs  = torch.cat(all_probs)
    acc    = (preds == y[:len(preds)]).float().mean().item()
    counts = torch.bincount(preds, minlength=NUM_CLASSES).float()
    dom    = counts.max().item() / counts.sum().item()
    n_cls  = (counts > 0).sum().item()
    h_yx   = -(probs * torch.log(probs + 1e-8)).sum(1).mean().item()
    mg     = probs.mean(0)
    h_y    = -(mg * torch.log(mg + 1e-8)).sum().item()
    return {'acc': round(acc, 4), 'dom_pct': round(dom, 4),
            'n_classes': n_cls, 'h_y': round(h_y, 4),
            'h_yx': round(h_yx, 4), 'mi': round(h_y - h_yx, 4)}

def is_collapsed(r):
    return (r['dom_pct'] > 0.15 and r['n_classes'] < 50) or r['n_classes'] < 20

print('Helpers ready.')
print(f'Collapse criterion: dom_pct > 0.15 AND n_classes < 50, OR n_classes < 20')

In [ ]:
# ================================================================
# Step 4: TTA Methods
# V3 changes (2 only, 6-professor approved):
#   (1) lambda_init = 0.1  (was 0.5)  — V17.1 proves safe at lr<=0.1
#   (2) ACCUM = 2, TTA_BATCH = 64   — effective batch 128 for better H(Y)
# ================================================================
def run_method(X, y, method, lr):
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    model.train()
    freeze_except_bn(model)
    params = [p for p in model.parameters() if p.requires_grad]
    opt    = optim.SGD(params, lr=lr, momentum=0.9)
    init_params = {pn: p.data.clone()
                   for pn, p in model.named_parameters() if p.requires_grad}

    # BMIA-A (old): lambda_init=0.5, effective_batch=64 (original behaviour)
    lambda_old = 0.5

    # BMIA-A-V3 (new): lambda_init=0.1, effective_batch=128 (via ACCUM=2)
    lambda_v3 = 0.1

    step_size = TTA_BATCH * ACCUM   # advance by effective batch each outer step

    for i in range(0, X.size(0), step_size):

        # ── TENT and BMIA-A use single batch (TTA_BATCH=64) ─────
        if method in ('tent', 'bmia_adaptive'):
            xb = X[i:i + TTA_BATCH]
            if xb.size(0) < 4: continue

            opt.zero_grad()
            probs = torch.softmax(model(xb), 1)
            ent   = -(probs * torch.log(probs + 1e-8)).sum(1)

            if method == 'tent':
                loss = ent.mean()

            else:  # bmia_adaptive (old, λ_init=0.5)
                mg     = probs.mean(0)
                h_y    = -(mg * torch.log(mg + 1e-8)).sum()
                anchor = sum(((p - init_params[pn])**2).sum()
                             for pn, p in model.named_parameters() if pn in init_params)
                loss   = ent.mean() - h_y + lambda_old * anchor

            loss.backward()
            opt.step()

            # Lambda update for bmia_adaptive
            if method == 'bmia_adaptive':
                with torch.no_grad():
                    dom = (torch.bincount(probs.argmax(1),
                            minlength=NUM_CLASSES).float().max() / xb.size(0)).item()
                lambda_old = (min(10.0, lambda_old * 1.1) if dom > 0.10
                              else max(0.01, lambda_old * 0.95))

        # ── BMIA-A-V3: effective batch = TTA_BATCH * ACCUM ──────
        elif method == 'bmia_adaptive_v3':
            # Collect ACCUM mini-batches and combine for better H(Y) estimate
            chunks = [X[i + s*TTA_BATCH : i + (s+1)*TTA_BATCH]
                      for s in range(ACCUM)]
            chunks = [c for c in chunks if c.size(0) >= 4]
            if not chunks: continue
            xb_all = torch.cat(chunks, dim=0)   # shape: [128, C, H, W]

            opt.zero_grad()
            probs_all = torch.softmax(model(xb_all), 1)
            ent_all   = -(probs_all * torch.log(probs_all + 1e-8)).sum(1)

            # H(Y) estimated over 128 samples — more accurate marginal
            mg     = probs_all.mean(0)
            h_y    = -(mg * torch.log(mg + 1e-8)).sum()
            anchor = sum(((p - init_params[pn])**2).sum()
                         for pn, p in model.named_parameters() if pn in init_params)
            loss   = ent_all.mean() - h_y + lambda_v3 * anchor

            loss.backward()
            opt.step()

            # Collapse monitor: dom% over the combined batch
            with torch.no_grad():
                dom = (torch.bincount(probs_all.argmax(1),
                        minlength=NUM_CLASSES).float().max() / xb_all.size(0)).item()
            lambda_v3 = (min(10.0, lambda_v3 * 1.1) if dom > 0.10
                         else max(0.01, lambda_v3 * 0.95))

    return get_metrics(model, X, y)

print('run_method ready.')
print('V3 key params: lambda_init=0.1, effective_batch=128 (ACCUM=2 x TTA_BATCH=64)')

In [ ]:
# ================================================================
# Step 5: Ablation — isolate each V3 improvement
# 4 variants at ABL_LR_LIST = [0.01, 0.05]
# ================================================================
# Variant definitions: (name, lambda_init, use_accum)
ABLATION_VARIANTS = [
    ('bmia_v3_base',    0.5, False),  # original BMIA-A (neither change)
    ('bmia_v3_accum',   0.5, True),   # + gradient accumulation only
    ('bmia_v3_lambda',  0.1, False),  # + reduced lambda only
    ('bmia_v3_both',    0.1, True),   # both = BMIA-A-V3
]

def run_ablation_v3(X, y, lr, lambda_init, use_accum):
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    model.train()
    freeze_except_bn(model)
    params = [p for p in model.parameters() if p.requires_grad]
    opt    = optim.SGD(params, lr=lr, momentum=0.9)
    init_params = {pn: p.data.clone()
                   for pn, p in model.named_parameters() if p.requires_grad}

    current_lambda = lambda_init
    eff_batch = TTA_BATCH * ACCUM if use_accum else TTA_BATCH

    for i in range(0, X.size(0), eff_batch):
        if use_accum:
            chunks = [X[i + s*TTA_BATCH : i + (s+1)*TTA_BATCH]
                      for s in range(ACCUM)]
            chunks = [c for c in chunks if c.size(0) >= 4]
            if not chunks: continue
            xb = torch.cat(chunks, dim=0)
        else:
            xb = X[i:i + TTA_BATCH]
            if xb.size(0) < 4: continue

        opt.zero_grad()
        probs = torch.softmax(model(xb), 1)
        ent   = -(probs * torch.log(probs + 1e-8)).sum(1)

        mg     = probs.mean(0)
        h_y    = -(mg * torch.log(mg + 1e-8)).sum()
        anchor = sum(((p - init_params[pn])**2).sum()
                     for pn, p in model.named_parameters() if pn in init_params)
        loss   = ent.mean() - h_y + current_lambda * anchor

        loss.backward()
        opt.step()

        # Collapse monitor (runs every optimizer step)
        with torch.no_grad():
            dom = (torch.bincount(probs.argmax(1),
                    minlength=NUM_CLASSES).float().max() / xb.size(0)).item()
        current_lambda = (min(10.0, current_lambda * 1.1) if dom > 0.10
                          else max(0.01, current_lambda * 0.95))

    return get_metrics(model, X, y)

print('run_ablation_v3 ready.')
print(f'Ablation variants: {[v[0] for v in ABLATION_VARIANTS]}')

In [ ]:
# ================================================================
# Step 6: Main Sweep — 3 methods × 4 lr × 5 corruptions = 60 runs
# ================================================================
all_results = []
t_total = time.time()

for corr in CORRUPTIONS:
    X, y = load_corruption(corr, severity=SEVERITY)
    print(f'\n=== {corr} ===')

    for lr in LR_LIST:
        for method in METHODS:
            t0  = time.time()
            r   = run_method(X, y, method=method, lr=lr)
            col = is_collapsed(r)
            print(f'  lr={lr:.3f} {method:<20} acc={r["acc"]:.4f} '
                  f'dom={r["dom_pct"]:.1%} cls={r["n_classes"]:3d} '
                  f'MI={r["mi"]:.2f} ({time.time()-t0:.0f}s)'
                  + (' *** COLLAPSED ***' if col else ''))
            all_results.append({
                'corruption': corr, 'severity': SEVERITY,
                'method': method, 'lr': lr,
                **r, 'collapsed': col
            })
            gc.collect(); torch.cuda.empty_cache()

    del X, y; gc.collect(); torch.cuda.empty_cache()

print(f'\nMain sweep done: {(time.time()-t_total)/60:.1f} min  |  {len(all_results)} results')

In [ ]:
# ================================================================
# Step 7: Ablation — 4 variants × 2 lr × 5 corruptions = 40 runs
# ================================================================
ablation_results = []
t_abl = time.time()

for corr in CORRUPTIONS:
    X, y = load_corruption(corr, severity=SEVERITY)
    print(f'\n=== {corr} ===')

    for abl_lr in ABL_LR_LIST:
        for name, lam_init, use_acc in ABLATION_VARIANTS:
            t0  = time.time()
            r   = run_ablation_v3(X, y, lr=abl_lr,
                                  lambda_init=lam_init, use_accum=use_acc)
            col = is_collapsed(r)
            print(f'  lr={abl_lr:.3f} {name:<22} '
                  f'λ_init={lam_init} accum={use_acc} '
                  f'acc={r["acc"]:.4f} dom={r["dom_pct"]:.1%} '
                  f'cls={r["n_classes"]:3d} ({time.time()-t0:.0f}s)'
                  + (' COLLAPSED' if col else ''))
            ablation_results.append({
                'corruption': corr, 'lr': abl_lr,
                'variant': name, 'lambda_init': lam_init, 'use_accum': use_acc,
                **r, 'collapsed': col
            })
            gc.collect(); torch.cuda.empty_cache()

    del X, y; gc.collect(); torch.cuda.empty_cache()

print(f'\nAblation done: {(time.time()-t_abl)/60:.1f} min  |  {len(ablation_results)} results')

In [ ]:
# ================================================================
# Step 8: Summary Tables
# ================================================================
print('='*80)
print('MAIN: AVERAGE ACCURACY (%) vs LR — averaged over 5 corruptions')
print('='*80)
print(f'\n{"Method":<22}', end='')
for lr in LR_LIST:
    print(f'  lr={lr:.3f}', end='')
print('  |  Avg')
print('-'*80)
for method in METHODS:
    accs = []
    print(f'{method:<22}', end='')
    for lr in LR_LIST:
        md  = [r for r in all_results if r['method']==method and r['lr']==lr]
        avg = np.mean([r['acc'] for r in md])*100 if md else 0
        accs.append(avg)
        print(f'  {avg:>6.1f}%', end='')
    print(f'  |  {np.mean(accs):.1f}%')

print()
print('='*80)
print('MAIN: COLLAPSE RATE vs LR')
print('='*80)
print(f'\n{"Method":<22}', end='')
for lr in LR_LIST:
    print(f'  lr={lr:.3f}', end='')
print()
print('-'*80)
for method in METHODS:
    print(f'{method:<22}', end='')
    for lr in LR_LIST:
        md = [r for r in all_results if r['method']==method and r['lr']==lr]
        nc = sum(1 for r in md if r['collapsed'])
        print(f'  {nc}/{len(md)}', end='')
    print()

print()
print('='*80)
print('V3 GAIN over old BMIA-A per lr')
print('='*80)
for lr in LR_LIST:
    old = np.mean([r['acc'] for r in all_results
                   if r['method']=='bmia_adaptive' and r['lr']==lr])*100
    new = np.mean([r['acc'] for r in all_results
                   if r['method']=='bmia_adaptive_v3' and r['lr']==lr])*100
    print(f'  lr={lr:.3f}:  old={old:.1f}%  v3={new:.1f}%  gain={new-old:+.1f}%')

print()
print('='*80)
print('ABLATION: λ_init and accumulation — per lr, avg over 5 corruptions')
print('='*80)
for abl_lr in ABL_LR_LIST:
    print(f'\n  lr = {abl_lr}')
    print(f'  {"Variant":<22} {"λ_init":>7} {"Accum":>6} {"Avg Acc":>9} {"Collapses":>11}')
    print('  ' + '-'*60)
    for name, lam_init, use_acc in ABLATION_VARIANTS:
        md  = [r for r in ablation_results if r['variant']==name and r['lr']==abl_lr]
        avg = np.mean([r['acc'] for r in md])*100 if md else 0
        nc  = sum(1 for r in md if r['collapsed'])
        print(f'  {name:<22} {lam_init:>7} {str(use_acc):>6} {avg:>8.1f}% {nc:>3}/{len(md)}')

In [ ]:
# ================================================================
# Step 9: Save JSON
# ================================================================
save_data = {
    'experiment': 'accuracy_mode',
    'model': 'ResNet-18 (CIFAR-100, seed=42)',
    'clean_acc': clean_acc,
    'lr_list': LR_LIST,
    'severity': SEVERITY,
    'corruptions': CORRUPTIONS,
    'v3_changes': [
        'lambda_init=0.1 (was 0.5) — V17.1 confirms safe at lr<=0.1',
        'gradient_accumulation ACCUM=2, TTA_BATCH=64 → effective_batch=128',
    ],
    'ablation_variants': [
        {'name': name, 'lambda_init': lam, 'use_accum': acc}
        for name, lam, acc in ABLATION_VARIANTS
    ],
    'abl_lr_list': ABL_LR_LIST,
    'main_results': all_results,
    'ablation_results': ablation_results,
}
with open('accuracy_mode_results.json', 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print('Saved: accuracy_mode_results.json')

# Print full JSON for copy-paste
with open('accuracy_mode_results.json') as f:
    print(f.read())